In [1]:
import akshare as ak
import pandas as pd
from datetime import datetime


def get_hk_stock_data(symbol: str):
    """
    获取港股数据，仅保留交易日
    """
    try:
        # 获取数据
        stock_data = ak.stock_hk_daily(symbol=symbol, adjust="qfq")
        
        if stock_data.empty:
            print(f"警告：{symbol} 返回空数据")
            return pd.DataFrame()
        
        print(f"{symbol} 原始数据 shape: {stock_data.shape}")
        
        # 确保 date 是 datetime 类型
        if 'date' in stock_data.columns:
            stock_data['date'] = pd.to_datetime(stock_data['date'], errors='coerce')
        else:
            print(f"错误：{symbol} 数据中无 'date' 列")
            return pd.DataFrame()
        
        # 筛选所需列
        keep_cols = ['date', 'open', 'high', 'low', 'close', 'volume']
        available_cols = [c for c in keep_cols if c in stock_data.columns]
        data = stock_data[available_cols].copy()
        
        # 日期范围过滤
        start_date = pd.to_datetime("2024-06-01")
        end_date = pd.to_datetime("2026-08-31")
        
        data = data[(data['date'] >= start_date) & (data['date'] <= end_date)]
        
        if data.empty:
            print(f"警告：{symbol} 在 {start_date.date()} ~ {end_date.date()} 区间无数据")
            return pd.DataFrame()
        
        # 设置索引并排序
        data = data.set_index('date').sort_index()
        
        # --- 修改处：不再进行 reindex(full_index).ffill() ---
        # 直接在原始交易日数据上添加 Ticker
        data['Ticker'] = symbol.zfill(5) 
        
        print(f"{symbol} 处理后 shape: {data.shape} (仅含交易日)")
        return data
    
    except Exception as e:
        print(f"获取 {symbol} 失败: {str(e)}")
        return pd.DataFrame()


def save_multiple_hk_stocks(tickers, filename):
    all_data = []
    for ticker in tickers:
        df = get_hk_stock_data(ticker)
        if not df.empty:
            all_data.append(df)
        print(f"处理完成 {ticker}, 交易日数: {len(df)}")
    
    if not all_data:
        print("所有股票均无数据，无法保存")
        return
    
    final_df = pd.concat(all_data)
    
    # 把日期索引转为普通列，命名为 'date'
    final_df = final_df.reset_index(names='date')
    
    # 日期格式统一（保持你的风格）
    final_df['date'] = final_df['date'].dt.strftime('%Y/%m/%d')
    
    # 保存
    final_df.to_csv(filename, encoding="utf-8-sig", index=False)
    print(f"所有数据已保存至: {filename}")
    print(f"总行数: {len(final_df)}, 列名: {final_df.columns.tolist()}")

In [2]:
import pandas as pd
import re
import os
OUTPUT_RAW_DIR = "data/raw"
INPUT_DIR = "data/weibo_hot_history"
# 腾讯控股 (00700.HK)
words_tencent = [
    # 主词
    '腾讯'
]

# 阿里巴巴 (09988.HK)
words_alibaba = [
    '阿里',
    '天猫',
    '618',
    '菜鸟'
]

# 小米集团 (01810.HK)
words_xiaomi = [
    # 主词
    '小米',
    # 核心产品
    'SU7',
    # 人物
    '雷军'
]

def parse_md_file(filename):
    """解析单个md文件"""
    with open(filename, "r", encoding="utf-8") as f:
        text = f.read()
    #提取日期
    pattern = r'(\d{4}-\d{2}-\d{2})'          # 匹配 YYYY-MM-DD
    
    match = re.search(pattern, filename)
    if match:
        date = match.group(1)
    else:
        print("未找到日期")
        
    # 正则匹配每一条热搜
    pattern = re.compile(r"(\d+)\.\s+\[(.*?)\]\(.*?\)\s+`(.*?)`\s+-\s+(\d+)")
    matches = pattern.findall(text)

    data = []
    for rank, keyword, category, hot_value in matches:
        hot_value_int = int(hot_value)
        # 跳过 hot_value 为 0 的条目
        if hot_value_int == 0:
            continue
        #忽略大小写
        keyword_lower   = keyword.lower()
        category_lower  = category.lower()
        
        is_tencent = any(word.lower() in keyword_lower or word.lower() in category_lower 
                         for word in words_tencent)
        
        is_alibaba = any(word.lower() in keyword_lower or word.lower() in category_lower 
                        for word in words_alibaba)
        
        is_xiaomi = any(word.lower() in keyword_lower or word.lower() in category_lower 
                        for word in words_xiaomi)
        data.append({
            "rank": int(rank),
            "keyword": keyword,
            "category": category,
            "hot_value": int(hot_value),
            "date": date,
            "is_tencent": is_tencent,
            "is_alibaba": is_alibaba,
            "is_xiaomi": is_xiaomi
        })

    return data

    
def parse_all_files(folder_path):
    """批量解析文件夹下所有md文件"""
    all_data = []
    for file in os.listdir(folder_path):
        if file.endswith(".md"):
            filepath = os.path.join(folder_path, file)
            # print(f"Parsing {filepath} ...")
            all_data.extend(parse_md_file(filepath))
    df = pd.DataFrame(all_data)
    return df

In [3]:
import pandas as pd
from transformers import pipeline
import torch
import os
from sklearn.model_selection import train_test_split

# 配置
MODEL_NAME = "IDEA-CCNL/Erlangshen-Roberta-110M-Sentiment"
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

EWMA_SPAN = 7
OUTPUT_RAW_DIR = "data/raw"
OUTPUT_PROCESSED_DIR = "data/processed"
os.makedirs(OUTPUT_PROCESSED_DIR, exist_ok=True)

# 加载情感模型
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    device=DEVICE,
    dtype=torch.float16 if DEVICE != "cpu" else None
)

def get_sentiment_score(text):
    """返回 [-1, 1] 分数"""
    if pd.isna(text) or not text.strip():
        return 0.0
    try:
        result = sentiment_pipeline(text, truncation=True, max_length=512)[0]
        if '1' in result['label'] or 'positive' in result['label'].lower():
            return result['score']
        else:
            return -result['score']
    except Exception as e:
        print(f"情感分析出错: {text[:30]}... → {e}")
        return 0.0




Using device: mps


Device set to use mps


In [4]:
# 主程序
if __name__ == "__main__":
    #1.下载历史股价并保存
    tickers = ["00700", "09988", "01810"]    #股票号码
    save_multiple_hk_stocks(tickers, "data/hk_stocks.csv")

    #2.根据关键词，解析热搜并保存
    df = parse_all_files(INPUT_DIR)
    df_tencent = df[df["is_tencent"]].drop(columns=["is_tencent", "is_alibaba", "is_xiaomi"])
    df_alibaba = df[df["is_alibaba"]].drop(columns=["is_tencent", "is_alibaba", "is_xiaomi"])
    df_xiaomi = df[df["is_xiaomi"]].drop(columns=["is_tencent", "is_alibaba", "is_xiaomi"])

    #过滤掉长度小于4的热搜词条
    df_tencent = df_tencent[df_tencent['keyword'].str.len() >= 4]
    df_alibaba = df_alibaba[df_alibaba['keyword'].str.len() >= 4]
    df_xiaomi = df_xiaomi[df_xiaomi['keyword'].str.len() >= 4]
    os.makedirs(OUTPUT_RAW_DIR, exist_ok=True)
    os.makedirs(OUTPUT_PROCESSED_DIR, exist_ok=True)

    df_tencent.to_csv(os.path.join(OUTPUT_RAW_DIR, "tencent.csv"), index=False, encoding="utf-8-sig")
    df_alibaba.to_csv(os.path.join(OUTPUT_RAW_DIR, "alibaba.csv"), index=False, encoding="utf-8-sig")
    df_xiaomi.to_csv(os.path.join(OUTPUT_RAW_DIR, "xiaomi.csv"),  index=False, encoding="utf-8-sig")

    print("已保存tencent.csv, alibaba.csv, xiaomi.csv, path = " + OUTPUT_RAW_DIR)


    #3.利用现有模型生成伪标签并划分测试集
    
    THRESHOLD = 0.7    # 引入置信度阈值，过滤掉模型不确定的“模糊样本”
    print("\n开始生成伪标签并划分测试集...")
    train_output_path = os.path.join(OUTPUT_PROCESSED_DIR, "train_data.csv")
    test_output_path = os.path.join(OUTPUT_PROCESSED_DIR, "test_data.csv")
    # 检查测试集是否存在，防止人工标注的测试集被覆盖
    if os.path.exists(train_output_path) and os.path.exists(test_output_path):
        print(f"训练集和测试集已存在，跳过生成")
    else:
        # 1）合并所有原始文本数据
        raw_texts = pd.concat([
            df_tencent['keyword'],
            df_alibaba['keyword'],
            df_xiaomi['keyword']
        ]).reset_index(drop=True)
        
        # 2）利用现有的 pipeline 批量打分
        print("正在批量计算情感分数...")
        scores = [get_sentiment_score(text) for text in raw_texts]
        
        # 3）设定阈值生成伪标签 (Pseudo-Labels)
        pseudo_labels = []
        valid_indices = []  # 记录有效样本的索引
        
        for i, score in enumerate(scores):
            if score > THRESHOLD:
                pseudo_labels.append(1)
                valid_indices.append(i)
            elif score < -THRESHOLD:
                pseudo_labels.append(0)
                valid_indices.append(i)
        
        # 4）组装成用于训练的 DataFrame（只保留有效样本）
        df_for_split = pd.DataFrame({
            'text': raw_texts.iloc[valid_indices],  # 只取有效索引的文本
            'label': pseudo_labels
        })
        
        print(f"原始数据共 {len(raw_texts)} 条，过滤模糊样本后，剩余高质量伪标签数据 {len(df_for_split)} 条。")
        
        #4. 划分训练集和测试集 (80% 训练, 20% 测试)
        train_df, test_df = train_test_split(
            df_for_split, 
            test_size=0.2, 
            random_state=42,      # 固定随机种子，保证每次划分结果一致
            stratify=pseudo_labels # 分层抽样，保证训练集和测试集的正负样本比例一致
        )
        # 6） 保存测试集到 CSV
        train_df.to_csv(train_output_path, index=False, encoding="utf-8-sig")
        print(f"训练集已保存至: {train_output_path} ({len(train_df)} 条)")
        test_df.to_csv(test_output_path, index=False, encoding="utf-8-sig")
        print(f"测试集已保存至: {test_output_path} ({len(test_df)} 条)")
    
        print(f"测试集生成完毕！共 {len(test_df)} 条数据。")
        print(f"文件已保存至: {test_output_path}")
        print(f"数据预览:\n{test_df.head()}")

00700 原始数据 shape: (5474, 6)
00700 处理后 shape: (552, 6) (仅含交易日)
处理完成 00700, 交易日数: 552
09988 原始数据 shape: (1677, 6)
09988 处理后 shape: (552, 6) (仅含交易日)
处理完成 09988, 交易日数: 552
01810 原始数据 shape: (2020, 6)
01810 处理后 shape: (552, 6) (仅含交易日)
处理完成 01810, 交易日数: 552
所有数据已保存至: data/hk_stocks.csv
总行数: 1656, 列名: ['date', 'open', 'high', 'low', 'close', 'volume', 'Ticker']
已保存tencent.csv, alibaba.csv, xiaomi.csv, path = data/raw

开始生成伪标签并划分测试集...
训练集和测试集已存在，跳过生成


In [8]:
#使用Soft Label（软标签）实现自学习微调
import pandas as pd
import torch
import numpy as np
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline
)
from torch.utils.data import Dataset
from torch.nn import BCEWithLogitsLoss
import warnings
warnings.filterwarnings("ignore")
#自定义 Dataset
class StockSentimentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length, is_train=True):
        self.tokenizer = tokenizer
        self.texts = dataframe['text'].tolist()
        self.max_length = max_length
        self.is_train = is_train
        if is_train:
            self.labels = dataframe['soft_label'].tolist()
        else:
            self.labels = dataframe['label'].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float if self.is_train else torch.long)
        }
        
#生成软标签        
def get_soft_label(text):
    if pd.isna(text) or not str(text).strip():
        return 0.5
    try:
        result = sentiment_pipeline(str(text))[0]
        score = result['score']
        if '1' in result['label'] or 'positive' in result['label'].lower():
            return score
        else:
            return 1 - score
    except:
        return 0.5

#评估函数
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

#Soft Label Trainer
class SoftLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        if labels.dtype == torch.float:          # 训练集：软标签
            pos_logits = logits[:, 1]
            loss_fct = BCEWithLogitsLoss()
            loss = loss_fct(pos_logits, labels)
        else:                                   # 测试集：硬标签
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss
#1）基础配置 
MODEL_NAME = "IDEA-CCNL/Erlangshen-Roberta-110M-Sentiment"
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 4                          # 训练轮数
LEARNING_RATE = 2e-5                # 低学习率

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"正在使用设备: {DEVICE}")

#2）加载数据
print("正在加载数据...")
train_df = pd.read_csv('data/processed/train_data.csv')
test_df = pd.read_csv('data/processed/test_data.csv')
print(f"训练集规模: {len(train_df)} 条")
print(f"测试集规模: {len(test_df)} 条")

#3）为训练集生成软标签
print("正在使用原始模型生成软标签...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    device=0 if DEVICE == "cuda" else -1,
    truncation=True,
    max_length=512
)

train_df['soft_label'] = train_df['text'].apply(get_soft_label)
print("软标签生成完成，示例：")
print(train_df[['text', 'label', 'soft_label']].head())

#4）初始化模型和数据
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_dataset = StockSentimentDataset(train_df, tokenizer, MAX_LENGTH, is_train=True)
test_dataset = StockSentimentDataset(test_df, tokenizer, MAX_LENGTH, is_train=False)

print(f"正在加载预训练模型: {MODEL_NAME} ...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#5）计算 Epoch 0（原始模型）的指标，用于对比
print("\n正在评估原始模型（Epoch 0）...")
temp_trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir='./tmp_pre', report_to="none", per_device_eval_batch_size=BATCH_SIZE),
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
pre_results = temp_trainer.evaluate()

epoch0 = {
    'Epoch': 0,
    'Training Loss': None,
    'Validation Loss': pre_results.get('eval_loss'),
    'Accuracy': pre_results.get('eval_accuracy'),
    'F1': pre_results.get('eval_f1'),
    'Precision': pre_results.get('eval_precision'),
    'Recall': pre_results.get('eval_recall'),
}


#6）训练参数配置
training_args = TrainingArguments(
    output_dir='./results_soft',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",    #自动选择 validation loss最小的模型
    greater_is_better=False,
    report_to="none",
    dataloader_pin_memory=False   # 消除警告
)

#7）开始训练
trainer = SoftLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n开始 Soft Label 微调...")
trainer.train()

#8）汇总所有 Epoch 结果
print("\n" + "="*90)
print("各 Epoch 指标汇总（含原始模型 Epoch 0）")
print("="*90)

history = trainer.state.log_history
results_list = [epoch0]

for i in range(1, EPOCHS + 1):
    # 提取验证指标
    eval_log = next((log for log in history if log.get('epoch') == float(i) and 'eval_loss' in log), None)
    
    # 提取该 epoch 最后一次训练 loss
    train_losses = [log['loss'] for log in history 
                    if 'loss' in log and 'eval_loss' not in log and log.get('epoch', 0) <= float(i)]
    train_loss = train_losses[-1] if train_losses else None

    if eval_log:
        results_list.append({
            'Epoch': i,
            'Training Loss': train_loss,
            'Validation Loss': eval_log.get('eval_loss'),
            'Accuracy': eval_log.get('eval_accuracy'),
            'F1': eval_log.get('eval_f1'),
            'Precision': eval_log.get('eval_precision'),
            'Recall': eval_log.get('eval_recall'),
        })

df_results = pd.DataFrame(results_list)
print(df_results.to_string(index=False, float_format="%.6f"))
print("="*90)

#9）最终评估与保存
print("\n正在进行最终评估...")
final_results = trainer.evaluate()
print("最终测试结果:", final_results)

save_path = "./fine_tuned_sentiment_model_soft"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"\n模型已保存至: {save_path}")

正在使用设备: mps
正在加载数据...
训练集规模: 1408 条
测试集规模: 352 条
正在使用原始模型生成软标签...


Device set to use cpu


软标签生成完成，示例：
             text  label  soft_label
0  腾讯三季度净利润631.3亿      1    0.862248
1   天猫618惊喜红包真的好大      1    0.997997
2     腾讯回应余承东鸿蒙喊话      1    0.806279
3       雷军壁纸 逆天改命      1    0.915081
4         小米澎湃OS3      1    0.992845
正在加载预训练模型: IDEA-CCNL/Erlangshen-Roberta-110M-Sentiment ...

正在评估原始模型（Epoch 0）...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None, 'pad_token_id': 0}.



开始 Soft Label 微调...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.338600,0.198756,0.957386,0.972376,0.992481,0.953069
2,0.323100,0.176712,0.954545,0.970803,0.981550,0.960289
3,0.313100,0.171880,0.948864,0.967273,0.974359,0.960289
4,0.304900,0.181132,0.940341,0.962025,0.963768,0.960289



各 Epoch 指标汇总（含原始模型 Epoch 0）
 Epoch  Training Loss  Validation Loss  Accuracy       F1  Precision   Recall
     0            NaN         0.200147  0.968750 0.979817   0.996269 0.963899
     1       0.338600         0.198756  0.957386 0.972376   0.992481 0.953069
     2       0.323100         0.176712  0.954545 0.970803   0.981550 0.960289
     3       0.313100         0.171880  0.948864 0.967273   0.974359 0.960289
     4       0.304900         0.181132  0.940341 0.962025   0.963768 0.960289

正在进行最终评估...


最终测试结果: {'eval_loss': 0.17187994718551636, 'eval_accuracy': 0.9488636363636364, 'eval_f1': 0.9672727272727273, 'eval_precision': 0.9743589743589743, 'eval_recall': 0.9602888086642599, 'eval_runtime': 5.1877, 'eval_samples_per_second': 67.853, 'eval_steps_per_second': 2.12, 'epoch': 4.0}

模型已保存至: ./fine_tuned_sentiment_model_soft
